# Preparazione dei dataframe per la classificazione
Per alleggerire i notebook sulla classificazione, prepariamo separatamente i dataframe da utilizzare in questo notebook. Si organizza questo notebook come una semplice rassegna di notebook, mantenendo al minimo i commenti. Si contestualizzino quindi i dataframe nel modo in cui vengono utilizzati.

In [78]:
# importiamo i pacchetti necessari
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from  sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import metrics

import sys
sys.path.append('../..')

import src.class_funcs as fs

## Classificazione binaria

In [79]:
# importiamo il dataframe sviluppato per l'EDA
dataset_df = pd.read_csv('../../data/processed/dataset_EDA_processed.csv')
dataset_df.head()

,station_appa,date,hour,CO,NO2,O3,PM10,PM2.5,SO2,geometry_appa,...,elevation,temperature,precipitation,winds_spd,winds_dir,geometry_weather,power_area_50,power_area_1000,power_area_2500,week_day
0,Borgo Valsugana,2013-11-01,1,NaN,21.0,2.0,19.0,12.0,NaN,POINT (11.45389 46.05184),...,410,11.350,0.0,NaN,NaN,POINT (11.47747769 46.05804607),17.760273,42.946772,54.704690,friday
1,Borgo Valsugana,2013-11-01,2,NaN,18.0,2.0,18.0,11.0,NaN,POINT (11.45389 46.05184),...,410,11.525,0.0,NaN,NaN,POINT (11.47747769 46.05804607),16.362439,39.762416,52.679229,friday
2,Borgo Valsugana,2013-11-01,3,NaN,19.0,2.0,19.0,12.0,NaN,POINT (11.45389 46.05184),...,410,11.425,0.0,NaN,NaN,POINT (11.47747769 46.05804607),17.861264,41.283915,50.294214,friday
3,Borgo Valsugana,2013-11-01,4,NaN,17.0,2.0,20.0,11.0,NaN,POINT (11.45389 46.05184),...,410,11.075,0.0,NaN,NaN,POINT (11.47747769 46.05804607),14.669913,36.119279,45.105859,friday
4,Borgo Valsugana,2013-11-01,5,NaN,18.0,2.0,21.0,13.0,NaN,POINT (11.45389 46.05184),...,410,10.950,0.0,NaN,NaN,POINT (11.47747769 46.05804607),16.969367,39.626135,48.951222,friday


per la classificazione binaria vogliamo usare le seguenti features:
- 'station_appa' in formato one-hot
- 'elevation'
- 'day' : processione ordinata di giorni dal primo dato
- 'week_day' in formato one-hot
- 'hour' 
- per gli inquinanti 'PM10', 'NO2', 'AQI' riferiti a 1, 2 e 3 ore prima -> non vogliamo NaN, quindi togliamo gli inquinanti non misurati da tutte le stazioni.
- per il meteo 'temperature' e 'winds_spd' riferiti ad 1 ora prima e 'precipitation' riferita a 5 ore prima
- per i consumi elettrici 'power_area_50' riferita a 1, 2 e 3 ore prima

come target mettiamo 1 = 'ok' se 'EAQI' è 'good' o 'fair' e 0 = 'not ok' se 'EAQI' è 'moderate', 'poor' o 'very poor'

NB: 'winds_spd' è molto correlato con gli inquinanti, ma capire come tenere conto di questo dato è molto complicato visto che spesso non ci sono rilevazioni. Per il momento lo inseriamo nella tabella. Vedremo successivamente se sostituire i 'NaN' o se eventualmente droppare la colonna.

In [80]:
# inseriamo in dataset_df una colonna 'day' che contiene il giorno del mese estratto dalla colonna 'date'
dataset_df['date'] = pd.to_datetime(dataset_df['date'])
dataset_df['day'] = (dataset_df["date"] - dataset_df["date"].min()).dt.days + 1

# feature istantanee
binary_class_df = dataset_df[['station_appa', 'elevation', 'day', 'week_day', 'hour']].copy()
binary_class_df = pd.get_dummies(binary_class_df, columns=['station_appa'], prefix='station', dtype=int)
binary_class_df = pd.get_dummies(binary_class_df, columns=['week_day'], prefix='', prefix_sep='', dtype=int)

# riordiniamo le colonne 
week_cols = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']
station_cols = [col for col in binary_class_df.columns if col.startswith('station_')]

binary_class_df = binary_class_df[station_cols + week_cols + ['elevation', 'day', 'hour']]

# aggiungiamo le colonne con gli inquinanti ed i consumi elettrici spostati di 1, 2 e 3 ore
pol_cols = ['PM10', 'NO2', 'AQI']
for h in range (1,4):
    binary_class_df[[col + f'_{h}' for col in pol_cols]] = dataset_df.groupby('station_appa')[pol_cols].shift(h)
    binary_class_df[f'power_area_50_{h}'] = dataset_df.groupby('station_appa')['power_area_50'].shift(h)

# aggiungiamo le colonne con il meteo
binary_class_df['temperature'] = dataset_df.groupby('station_appa')['temperature'].shift(1)
binary_class_df['winds_spd'] = dataset_df.groupby('station_appa')['winds_spd'].shift(1)
binary_class_df['precipitation'] = dataset_df.groupby('station_appa')['precipitation'].shift(5)

# aggiungiamo il target
binary_class_df['target'] = ((dataset_df['EAQI'] == 'good') | (dataset_df['EAQI'] == 'fair')).astype(int)

# eliminiamo le righe che abbiamo appena agginto con valori NaN
binary_class_df.dropna(subset='precipitation', inplace=True)
binary_class_df = binary_class_df.reset_index(drop=True)

binary_class_df.to_csv("../../data/processed/dataset_binary_class_processed.csv", index=False)

binary_class_df.head()

,station_Borgo Valsugana,station_Monte Gaza,station_Parco S. Chiara,station_Piana Rotaliana,station_Riva del Garda,station_Rovereto,station_Via Bolzano,monday,tuesday,wednesday,...,AQI_2,power_area_50_2,PM10_3,NO2_3,AQI_3,power_area_50_3,temperature,winds_spd,precipitation,target
0,1,0,0,0,0,0,0,0,0,0,...,22.0,14.669913,19.0,19.0,24.0,17.861264,10.950,NaN,0.0,1
1,1,0,0,0,0,0,0,0,0,0,...,26.0,16.969367,20.0,17.0,22.0,14.669913,11.000,NaN,0.0,1
2,1,0,0,0,0,0,0,0,0,0,...,26.0,15.329278,21.0,18.0,26.0,16.969367,10.925,NaN,0.0,1
3,1,0,0,0,0,0,0,0,0,0,...,32.0,17.519438,19.0,16.0,26.0,15.329278,10.950,NaN,0.0,1
4,1,0,0,0,0,0,0,0,0,0,...,22.0,17.446178,24.0,16.0,32.0,17.519438,11.550,NaN,0.0,1
